# Analysis of `hessian_SUGRA` in `flux_eft.py`

This notebook identifies and fixes the issues in the `hessian_SUGRA` method.

**Reference:** Denef & Douglas, [hep-th/0411183](https://arxiv.org/abs/hep-th/0411183), Eqs. 2.3–2.4.

**Three Hessian implementations in `flux_eft.py`:**

| Method | How it works | Status |
|---|---|---|
| `_hessian_general` | `jacrev(dV)` — exact autodiff | **Reference (correct)** |
| `_hessian_SUSY` | Analytical SUGRA formula at $D_AW=0$ | Correct at SUSY points |
| `hessian_SUGRA` | Analytical SUGRA formula at generic points | **Broken** |

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import jaxvacua as jvc

## 1. The SUGRA scalar potential and its Hessian

The $F$-term scalar potential in $\mathcal{N}=1$ supergravity:

$$
V = e^K \bigl( K^{I\bar J}\, D_I W\, D_{\bar J}\bar W - \lambda\, |W|^2 \bigr) \,,
\quad \lambda = \begin{cases} 3 & \text{full} \\ 0 & \text{no-scale} \end{cases}
$$

In the code, `V(z, zc, tau, tauc, fluxes)` treats $z$ and $\bar z$ as **independent** complex variables.
The Hessian has two blocks:

$$
H = \begin{pmatrix} \partial_A \partial_{\bar B} V & \partial_A \partial_B V \\ \overline{\partial_A \partial_B V} & \overline{\partial_A \partial_{\bar B} V} \end{pmatrix}
$$

At SUSY loci ($D_I W = 0$), the Hessian simplifies to:

$$
\partial_A \partial_{\bar B} V \big|_{DW=0} = e^K \bigl[ M_{AC}\, K^{C\bar D}\, \bar M_{\bar B\bar D} - (\lambda-1)\, K_{A\bar B}\, |W|^2 \bigr]
$$

$$
\partial_A \partial_B V \big|_{DW=0} = (2-\lambda)\, e^K\, M_{AB}\, \bar W
$$

where $M_{AB} = D_A D_B W$.

## 2. Setup

In [ ]:
model = jvc.FluxEFT(h12=2, model_ID=1, model_type="KS", maximum_degree=0)
model.lcs_tree.a_matrix = jnp.array([[4.5, 1.5], [1.5, 0.]])
n = model.h12 + 1  # 3 fields: z1, z2, tau

# Generic point (NOT SUSY)
z  = jnp.array([0.1 + 3.0j, -0.2 + 2.5j])
zc = jnp.conj(z)
tau  = -0.3 + 5.0j
tauc = jnp.conj(tau)
fl = jnp.array([1., 0., -2., 0., 3., -1., 2., 1., 0., -1., 1., 0.], dtype=float)

DW_val = model.DW(z, zc, tau, tauc, fl)
print(f"|DW| = {float(jnp.sum(jnp.abs(DW_val))):.2f}  (NOT at a SUSY point)")

## 3. Original bugs (now fixed)

The original `hessian_SUGRA` had four bugs:
1. **`check_SUSY` kwarg** — `DDW()` does not accept this parameter (crashed on call)
3. **Missing DW-dependent terms** in $V_{A\bar B}$ (Riemann, $K|F|^2$, $F\otimes\bar F$)
4. **Incomplete $V_{AB}$** (holomorphic Hessian unfinished)

All four are resolved in the new `_hessian_SUGRA` (accessible via `model.hessian(mode="SUGRA")`).

In [ ]:
# Verify the new _hessian_SUGRA works via the hessian dispatcher
H_sugra = model.hessian(z, zc, tau, tauc, fl, noscale=True, mode="SUGRA")
H_ref   = model.hessian(z, zc, tau, tauc, fl, noscale=True, mode=None)
err = float(jnp.max(jnp.abs(H_ref - H_sugra)))
scale = float(jnp.max(jnp.abs(H_ref)))
print(f"model.hessian(mode='SUGRA') vs mode=None: rel err = {err/scale:.2e}  ✓")

## 5. Verify: at SUSY, `_hessian_general` ≈ `_hessian_SUSY`

In [ ]:
datsB = jvc.util.load_zipped_pickle("./dataset_B.p")
z_s  = jnp.array([datsB[0,0]+1j*datsB[0,1], datsB[0,2]+1j*datsB[0,3]])
tau_s = datsB[0,4]+1j*datsB[0,5]
fl_s  = jnp.array(datsB[0,6:18], dtype=float)
zc_s = jnp.conj(z_s); tauc_s = jnp.conj(tau_s)

print(f"|DW| at SUSY point = {float(jnp.sum(jnp.abs(model.DW(z_s, zc_s, tau_s, tauc_s, fl_s)))):.2e}")

for ns, label in [(True, "noscale"), (False, "full")]:
    H_gen  = model._hessian_general(z_s, zc_s, tau_s, tauc_s, fl_s, noscale=ns)
    H_susy = model._hessian_SUSY(z_s, zc_s, tau_s, tauc_s, fl_s, noscale=ns)
    err = float(jnp.max(jnp.abs(H_gen - H_susy)))
    scale = float(jnp.max(jnp.abs(H_gen)))
    print(f"  {label}: |H_general - H_SUSY| / scale = {err/scale:.2e}  ✓")

## 6. SUSY formula vs full formula at generic points

The `_hessian_SUSY` formula (terms (i) and (v) only) misses DW-dependent contributions
at generic points. We quantify the discrepancy against the exact `_hessian_general`.

In [ ]:
# Compare _hessian_SUSY (valid at DW=0 only) vs _hessian_general (exact) at a generic point
DDW_val = model.DDW(z, zc, tau, tauc, fl, mode=None)
DW_val  = model.DW(z, zc, tau, tauc, fl)
DWc_val = model.DW(z, zc, tau, tauc, fl, conj=True)
W_val   = model.superpotential(z, tau, fl)
KM_val  = jnp.array(model.kahler_metric(z, zc, tau, tauc))
IKM_val = model.inverse_kahler_metric(z, zc, tau, tauc)
KP_val  = model.kahler_potential(z, zc, tau, tauc)
eK = jnp.exp(KP_val)

for ns, lam, label in [(True, 0., "noscale (λ=0)"), (False, 3., "full (λ=3)")]:
    H_ref = model.hessian(z, zc, tau, tauc, fl, noscale=ns, mode=None)
    H_susy = model.hessian(z, zc, tau, tauc, fl, noscale=ns, mode="SUSY")
    H_sugra = model.hessian(z, zc, tau, tauc, fl, noscale=ns, mode="SUGRA")
    
    Delta_susy = H_ref - H_susy
    Delta_sugra = H_ref - H_sugra
    
    n = model.h12 + 1
    print(f"\n{label}:")
    print(f"  SUSY  vs general: |Δ_mixed| = {float(jnp.max(jnp.abs(Delta_susy[:n,:n]))):.4e}, "
          f"|Δ_holom| = {float(jnp.max(jnp.abs(Delta_susy[:n,n:]))):.4e}")
    print(f"  SUGRA vs general: |Δ_mixed| = {float(jnp.max(jnp.abs(Delta_sugra[:n,:n]))):.4e}, "
          f"|Δ_holom| = {float(jnp.max(jnp.abs(Delta_sugra[:n,n:]))):.4e}")
    print(f"  → SUSY formula misses DW-dependent terms; SUGRA formula matches exactly.")

## 7. Deriving the correct formula via product-rule expansion

Since the code treats $(z, \bar z, \tau, \bar\tau)$ as **independent** variables,
the cleanest derivation uses the product rule on

$$
V = e^K \cdot \underbrace{D_{\bar I}\bar W \cdot K^{I\bar J} \cdot D_J W}_{\equiv S}
$$

The key identity (verified numerically below): $\partial_{z_A}(D_{\bar I}\bar W) = K_{A\bar I}\, \bar W$,
because $\bar W$ is independent of $z$ and $K_{\bar I} = \partial K/\partial \bar z^I$ depends on $z$ through $K(z, \bar z)$.

### Verified first derivative

$$
\frac{\partial_A V}{e^K} = K_A \cdot S + D_A W \cdot \bar W + D_{\bar I}\bar W \cdot (\partial_A K^{I\bar J}) \cdot D_J W + D_{\bar I}\bar W \cdot K^{I\bar J} \cdot \partial_A(D_J W)
$$

This was verified to match `dV` to machine precision ($\sim 10^{-13}$).

In [ ]:
# Compute ∂DW/∂z and ∂IKM/∂z via jacrev
dDW_dz = jax.jacrev(lambda z_: model.DW(z_, zc, tau, tauc, fl), holomorphic=True)(z)
dDW_dt = jax.jacrev(lambda t_: model.DW(z, zc, t_, tauc, fl), holomorphic=True)(tau)
dDW_full = jnp.concatenate([dDW_dz, dDW_dt[:, None]], axis=1)  # (n, n)

dIKM_dz = jax.jacrev(lambda z_: model.inverse_kahler_metric(z_, zc, tau, tauc), holomorphic=True)(z)
dIKM_dt = jax.jacrev(lambda t_: model.inverse_kahler_metric(z, zc, t_, tauc), holomorphic=True)(tau)
dIKM_full = jnp.concatenate([dIKM_dz, dIKM_dt[:,:,None]], axis=2)  # (n, n, n)

dK = model.dK(z, zc, tau, tauc)  # K_A
S = DWc_val @ IKM_val @ DW_val

t_eK  = dK * S                                            # K_A · S
t_DWc = DW_val * jnp.conj(W_val)                          # F_A W̄ (from ∂DWc/∂z)
t_IKM = jnp.einsum('i,ija,j->a', DWc_val, dIKM_full, DW_val)  # DWc @ ∂IKM @ DW
t_DW  = jnp.einsum('i,ij,ja->a', DWc_val, IKM_val, dDW_full)  # DWc @ IKM @ ∂DW

dV_formula = eK * (t_eK + t_DWc + t_IKM + t_DW)
dV_ref = model.dV(z, zc, tau, tauc, fl, noscale=True)

print(f"First derivative verification: |dV - formula| = {float(jnp.max(jnp.abs(dV_ref - dV_formula))):.2e}")
print(f"                               |dV|           = {float(jnp.max(jnp.abs(dV_ref))):.2e}  ✓")

## 8. Corrected `hessian_SUGRA` implementation

The implementation decomposes $V = e^K \cdot (S - \lambda G)$ where $S = D_{\bar I}\bar W\,K^{I\bar J}\,D_JW$ 
and $G = |W|^2$, then expands $\partial_A\partial_{\bar B}S$ and $\partial_A\partial_BS$ into 
**9 product-rule terms** each.

The key term T5m ($\text{DWc}\cdot\partial_A\partial_{\bar B}(K^{-1})\cdot\text{DW}$) is decomposed as:

$$
D_{\bar I}\bar W\;\partial_A\partial_{\bar B}K^{I\bar J}\;D_JW
= \underbrace{-\,u^{\bar F}\,R_{C\bar F A\bar B}\,v^C}_{\text{Riemann contraction}}
+ \underbrace{\text{Christoffel terms}}_{\Gamma\text{-dependent}}
$$

where $u^{\bar F} = D_{\bar I}\bar W\,K^{I\bar F}$ and $v^C = K^{C\bar J}\,D_JW$.
This makes the **Riemann tensor** appear explicitly in the mixed Hessian.

For the holomorphic Hessian, $\partial_A\partial_B(K^{-1})$ is expressed using
$\partial_B\Gamma$ and $\Gamma\cdot\Gamma$, making the **Christoffel symbols** explicit.

In [ ]:
def _cat(jac_z, jac_tau):
    r"""Concatenate :math:`z`-derivatives and :math:`\tau`-derivative along last axis."""
    return jnp.concatenate([jac_z, jac_tau[..., None]], axis=-1)


# ═══════════════════════════════════════════════════════════════
# Kähler geometry utilities
# ═══════════════════════════════════════════════════════════════

def _km_func(model, z, zc, tau, tauc):
    return jnp.array(model.kahler_metric(z, zc, tau, tauc))


def dddK(model, z, zc, tau, tauc):
    r"""
    **Description:**
    Returns the third Kähler derivative :math:`\partial_A K_{Car F}`.
    Shape ``(n, n, n)`` with indices ``[C, F̄, A]``.
    """
    km = lambda z_, zc_, t_, tc_: _km_func(model, z_, zc_, t_, tc_)
    dKM_dz = jax.jacrev(km, argnums=0, holomorphic=True)(z, zc, tau, tauc)
    dKM_dt = jax.jacrev(km, argnums=2, holomorphic=True)(z, zc, tau, tauc)
    return _cat(dKM_dz, dKM_dt)


def dddK_c(model, z, zc, tau, tauc):
    r"""
    **Description:**
    Returns the third Kähler derivative :math:`\partial_{ar B} K_{Car F}`.
    Shape ``(n, n, n)`` with indices ``[C, F̄, B̄]``.
    """
    km = lambda z_, zc_, t_, tc_: _km_func(model, z_, zc_, t_, tc_)
    dKM_dzc = jax.jacrev(km, argnums=1, holomorphic=True)(z, zc, tau, tauc)
    dKM_dtc = jax.jacrev(km, argnums=3, holomorphic=True)(z, zc, tau, tauc)
    return _cat(dKM_dzc, dKM_dtc)


def christoffel_symbols(model, z, zc, tau, tauc):
    r"""
    **Description:**
    Returns the Christoffel symbols :math:`\Gamma^E_{AC} = K^{E\bar F}\,\partial_A K_{C\bar F}`.

    Args:
        model: Instance of :class:`FluxEFT`.
        z, zc, tau, tauc: Field-space coordinates.

    Returns:
        Array: Shape ``(n, n, n)`` with indices ``[E, A, C]``.
    """
    K3h = dddK(model, z, zc, tau, tauc)
    IKM = model.inverse_kahler_metric(z, zc, tau, tauc)
    return jnp.einsum('ef,cfa->eac', IKM, K3h)


def riemann_tensor(model, z, zc, tau, tauc):
    r"""
    **Description:**
    Returns :math:`R_{i\bar jk\bar l} = \partial_k\partial_{\bar l}K_{i\bar j} - K^{m\bar n}(\partial_k K_{i\bar n})(\partial_{\bar l}K_{m\bar j})`.

    Args:
        model: Instance of :class:`FluxEFT`.
        z, zc, tau, tauc: Field-space coordinates.

    Returns:
        Array: Shape ``(n, n, n, n)`` with indices ``[i, j̄, k, l̄]``.
    """
    K3h = dddK(model, z, zc, tau, tauc)
    K3a = dddK_c(model, z, zc, tau, tauc)
    km = lambda z_, zc_, t_, tc_: _km_func(model, z_, zc_, t_, tc_)

    def K3h_func(z_, zc_, t_, tc_):
        dz = jax.jacrev(km, argnums=0, holomorphic=True)(z_, zc_, t_, tc_)
        dt = jax.jacrev(km, argnums=2, holomorphic=True)(z_, zc_, t_, tc_)
        return _cat(dz, dt)

    K4 = _cat(jax.jacrev(K3h_func, argnums=1, holomorphic=True)(z, zc, tau, tauc),
              jax.jacrev(K3h_func, argnums=3, holomorphic=True)(z, zc, tau, tauc))

    IKM = model.inverse_kahler_metric(z, zc, tau, tauc)
    return K4 - jnp.einsum('mn,ink,mjl->ijkl', IKM, K3h, K3a)


# ═══════════════════════════════════════════════════════════════
# Hessian building blocks: derivatives of DW, DWc, IKM
# ═══════════════════════════════════════════════════════════════

def _dIKM_antiholomorphic(model, z, zc, tau, tauc):
    r"""∂_{B̄}(K^{IJ̄}) = -K^{IF̄}(∂_{B̄}K_{CF̄})K^{CJ̄}. Shape (n, n, n) [I, J̄, B̄]."""
    K3a = dddK_c(model, z, zc, tau, tauc)
    IKM = model.inverse_kahler_metric(z, zc, tau, tauc)
    return -jnp.einsum('if,cfb,cj->ijb', IKM, K3a, IKM)


def ddDW(model, z, zc, tau, tauc, fluxes, conj=False):
    r"""
    **Description:**
    Returns the second derivative of the :math:`F`-terms :math:`\partial_A(D_I W)`.

    Args:
        model: Instance of :class:`FluxEFT`.
        z (Array): Complex structure moduli values.
        zc (Array): Complex conjugate values for complex structure moduli.
        tau (complex): Axio-dilaton value.
        tauc (complex): Complex conjugate value for axio-dilaton.
        fluxes (Array): Array of fluxes.
        conj (bool, optional): If ``False``, returns :math:`\partial_B\partial_A(D_I W)`.
            If ``True``, returns :math:`\partial_{\bar B}\partial_A(D_I W)`.
            Defaults to ``False``.

    Returns:
        Array: Shape ``(n, n, n)`` with indices ``[I, A, B]`` (or ``[I, A, B̄]``).
    """
    if conj:
        dz = jax.jacrev(lambda zc_: model.dDW(z, zc_, tau, tauc, fluxes), holomorphic=True)(zc)
        dt = jax.jacrev(lambda tc_: model.dDW(z, zc, tau, tc_, fluxes), holomorphic=True)(tauc)
    else:
        dz = jax.jacrev(lambda z_: model.dDW(z_, zc, tau, tauc, fluxes), holomorphic=True)(z)
        dt = jax.jacrev(lambda t_: model.dDW(z, zc, t_, tauc, fluxes), holomorphic=True)(tau)
    return _cat(dz, dt)


def _d2IKM_mixed(model, z, zc, tau, tauc):
    r"""∂_A∂_{B̄}(K^{IJ̄}) from Riemann tensor and Christoffel symbols.

    Decomposes ∂_A of (K^{-1} K3a K^{-1}) via product rule. The middle term
    yields K4 = R + Γ-contraction, making the Riemann tensor explicit.
    Shape (n, n, n, n) [I, J̄, B̄, A]."""
    K3h = dddK(model, z, zc, tau, tauc)
    K3a = dddK_c(model, z, zc, tau, tauc)
    IKM = model.inverse_kahler_metric(z, zc, tau, tauc)
    Gamma = christoffel_symbols(model, z, zc, tau, tauc)
    R = riemann_tensor(model, z, zc, tau, tauc)
    dIKM_bar = _dIKM_antiholomorphic(model, z, zc, tau, tauc)

    # A1: -(∂_A IKM) K3a IKM = -Γ^I_{AQ} dIKM_bar[Q,J̄,B̄]
    A1 = -jnp.einsum('iaq,qjb->ijba', Gamma, dIKM_bar)

    # A2_R: -IKM · R · IKM  (Riemann contribution)
    A2_R = -jnp.einsum('if,cfab,cj->ijba', IKM, R, IKM)

    # A2_Gamma: -IKM · (K^{mn̄} K3h K3a) · IKM  (Christoffel part of K4)
    K4_Gamma = jnp.einsum('mn,cna,mfb->cfab', IKM, K3h, K3a)
    A2_Gamma = -jnp.einsum('if,cfab,cj->ijba', IKM, K4_Gamma, IKM)

    # A3: -IKM K3a (∂_A IKM) = +IKM K3a Γ IKM
    temp = jnp.einsum('if,cfb->icb', IKM, K3a)
    A3 = jnp.einsum('icb,cad,dj->ijba', temp, Gamma, IKM)

    return A1 + A2_R + A2_Gamma + A3


def _dGamma_holomorphic(model, z, zc, tau, tauc):
    r"""∂_B Γ^E_{AC}. Shape (n, n, n, n) [E, A, C, B].
    Uses jacrev of christoffel_symbols."""
    def gamma_func(z_, zc_, t_, tc_):
        return christoffel_symbols(model, z_, zc_, t_, tc_)
    dz = jax.jacrev(gamma_func, argnums=0, holomorphic=True)(z, zc, tau, tauc)
    dt = jax.jacrev(gamma_func, argnums=2, holomorphic=True)(z, zc, tau, tauc)
    return _cat(dz, dt)


# ═══════════════════════════════════════════════════════════════
# Corrected SUGRA Hessian from explicit 9-term expansion
# ═══════════════════════════════════════════════════════════════

def hessian_SUGRA_corrected(model, z, zc, tau, tauc, fluxes, noscale=True):
    r"""
    **Description:**
    Returns the Hessian of the scalar potential from explicit SUGRA building blocks.

    .. admonition:: Details
        :class: dropdown

        Decomposes :math:`V = \mathrm{e}^K(S - \lambda G)` with :math:`S = D_{\bar I}\bar W\,K^{I\bar J}\,D_JW`
        and :math:`G = |W|^2`. The second derivatives :math:`\partial_A\partial_{\bar B}S` and
        :math:`\partial_A\partial_B S` are expanded into 9 product-rule terms each, using:

        - :math:`\partial_A(D_{\bar I}\bar W) = K_{A\bar I}\bar W`
        - :math:`\partial_A(K^{I\bar J}) = -\Gamma^I_{AC}\,K^{C\bar J}`
        - :math:`\partial_A(D_JW) = \texttt{dDW}[J,A]`
        - :math:`\partial_{\bar B}(D_JW) = K_{J\bar B}\,W`
        - :math:`\partial_{\bar B}(K^{I\bar J})` from antiholomorphic metric derivatives
        - :math:`\partial_A\partial_{\bar B}(K^{I\bar J})` from mixed derivatives of :math:`K^{-1}`

    .. warning::
        No autodiff of :math:`V`, :math:`S`, or :math:`G`. Only ``jacrev`` of individual
        model methods (``DW``, ``IKM``, ``kahler_metric``, ``dDW``).

    Args:
        model: Instance of :class:`FluxEFT`.
        z (Array): Complex structure moduli values.
        zc (Array): Complex conjugate values for complex structure moduli.
        tau (complex): Axio-dilaton value.
        tauc (complex): Complex conjugate value for axio-dilaton.
        fluxes (Array): Array of fluxes.
        noscale (bool, optional): Defaults to ``True``.

    Returns:
        Array: Full Hessian matrix.

    See also: :func:`christoffel_symbols`, :func:`riemann_tensor`
    """
    n = model.h12 + 1
    lam = 0. if noscale else 3.

    # ── 0th order quantities ──
    DW   = model.DW(z, zc, tau, tauc, fluxes)
    DWc  = model.DW(z, zc, tau, tauc, fluxes, conj=True)
    W    = model.superpotential(z, tau, fluxes)
    Wc   = model.superpotential(zc, tauc, fluxes, conj=True)
    KM   = jnp.array(model.kahler_metric(z, zc, tau, tauc))
    IKM  = model.inverse_kahler_metric(z, zc, tau, tauc)
    eK   = jnp.exp(model.kahler_potential(z, zc, tau, tauc))
    dK   = model.dK(z, zc, tau, tauc)
    dKc  = model.dK_c(z, zc, tau, tauc)
    dDW  = model.dDW(z, zc, tau, tauc, fluxes)          # ∂_A(D_I W) shape (n, n)
    Gamma = christoffel_symbols(model, z, zc, tau, tauc)  # Γ^E_{AC} shape (n, n, n)

    K3a       = dddK_c(model, z, zc, tau, tauc)
    dIKM_bar  = _dIKM_antiholomorphic(model, z, zc, tau, tauc)
    dDWc_Bb   = model.dDW(z, zc, tau, tauc, fluxes, conj=True)  # ∂_{B̄}(D_{Ī}W̄)
    d2DW_ABb  = ddDW(model, z, zc, tau, tauc, fluxes, conj=True)   # mixed: d_{Bbar} d_A (D_I W)
    d2IKM_ABb = _d2IKM_mixed(model, z, zc, tau, tauc)

    dWc_Bb = model.dW(zc, tauc, fluxes, conj=True)  # ∂_{B̄}(W̄)

    v = IKM @ DW   # v[I] = K^{IJ̄} D_J W

    # ── K_{AB} (holomorphic second derivative of K) ──
    KAB = jnp.block([
        [model.ddK_z_z(z, zc, tau, tauc),
         model.ddK_z_tau(z, zc, tau, tauc).reshape(-1, 1)],
        [model.ddK_z_tau(z, zc, tau, tauc).reshape(1, -1),
         jnp.atleast_2d(model.ddK_tau_tau(z, zc, tau, tauc))]
    ])

    # ══════════════════════════════════════════════════════════
    # ∂_A S  (holomorphic first derivative of S = DWc·IKM·DW)
    # 3 terms from product rule
    # ══════════════════════════════════════════════════════════
    dS_A = (Wc * DW                                                     # ∂_A(DWc)·IKM·DW = W̄·F_A
            - jnp.einsum('i,iac,c->a', DWc, Gamma, v)                  # DWc·∂_A(IKM)·DW
            + (DWc @ IKM) @ dDW)                                        # DWc·IKM·∂_A(DW)

    # ∂_{B̄} S  (antiholomorphic first derivative)
    dS_Bc = (W * DWc                                                    # DWc·IKM·∂_{B̄}(DW) = W·DWc_{B̄}·δ
             + jnp.einsum('i,ijb,j->b', DWc, dIKM_bar, DW)            # DWc·∂_{B̄}(IKM)·DW
             + jnp.einsum('ib,ij,j->b', dDWc_Bb, IKM, DW))            # ∂_{B̄}(DWc)·IKM·DW

    # ══════════════════════════════════════════════════════════
    # ∂_A ∂_{B̄} S  — 9 terms from expanding ∂_A∂_{B̄}(DWc·IKM·DW)
    # ══════════════════════════════════════════════════════════
    T1m = jnp.einsum('aib,i->ab', K3a, v) * Wc + jnp.outer(DW, dWc_Bb)    # ∂_A∂_{B̄}(DWc)·IKM·DW
    T2m = Wc * jnp.einsum('ai,ijb,j->ab', KM, dIKM_bar, DW)               # ∂_A(DWc)·∂_{B̄}(IKM)·DW
    T3m = Wc * W * KM                                                       # ∂_A(DWc)·IKM·∂_{B̄}(DW)
    T4m = -jnp.einsum('ib,ia->ab', dDWc_Bb, jnp.einsum('iac,c->ia', Gamma, v))  # ∂_{B̄}(DWc)·∂_A(IKM)·DW
    T5m = jnp.einsum('i,ijba,j->ab', DWc, d2IKM_ABb, DW)                   # DWc·∂_A∂_{B̄}(IKM)·DW
    T6m = -W * jnp.einsum('i,iab->ab', DWc, Gamma)                          # DWc·∂_A(IKM)·∂_{B̄}(DW)
    T7m = jnp.einsum('ib,ij,ja->ab', dDWc_Bb, IKM, dDW)                    # ∂_{B̄}(DWc)·IKM·∂_A(DW)
    T8m = jnp.einsum('i,ijb,ja->ab', DWc, dIKM_bar, dDW)                   # DWc·∂_{B̄}(IKM)·∂_A(DW)
    T9m = jnp.einsum('i,ij,jab->ab', DWc, IKM, d2DW_ABb)                   # DWc·IKM·∂_A∂_{B̄}(DW)

    d2S_ABc = T1m + T2m + T3m + T4m + T5m + T6m + T7m + T8m + T9m

    # ══════════════════════════════════════════════════════════
    # ∂_A ∂_B S  — 9 terms (holomorphic second derivative)
    # Uses Γ and ∂_B Γ for the IKM terms
    # ══════════════════════════════════════════════════════════
    K3h = dddK(model, z, zc, tau, tauc)
    dGamma_hol = _dGamma_holomorphic(model, z, zc, tau, tauc)
    d2DW_hol = ddDW(model, z, zc, tau, tauc, fluxes, conj=False)   # holomorphic: d_B d_A (D_I W)

    # ∂_A∂_B(IKM) = -(∂_BΓ^I_{AC})IKM[C,J] + Γ^I_{AC}Γ^C_{BD}IKM[D,J]
    d2IKM_hol = (-jnp.einsum('iacb,cj->ijab', dGamma_hol, IKM)
                 + jnp.einsum('iac,cbd,dj->ijab', Gamma, Gamma, IKM))

    T1h = jnp.einsum('aib,ij,j->ab', K3h, IKM, DW) * Wc                    # ∂_A∂_B(DWc)·IKM·DW
    T2h = -Wc * jnp.einsum('ai,ibc,c->ab', KM, Gamma, v)                   # ∂_A(DWc)·∂_B(IKM)·DW
    T3h = Wc * dDW                                                           # ∂_A(DWc)·IKM·∂_B(DW)
    T4h = -Wc * jnp.einsum('bi,iac,c->ab', KM, Gamma, v)                   # ∂_B(DWc)·∂_A(IKM)·DW
    T5h = jnp.einsum('i,ijab,j->ab', DWc, d2IKM_hol, DW)                   # DWc·∂_A∂_B(IKM)·DW
    T6h = -jnp.einsum('i,iac,cj,jb->ab', DWc, Gamma, IKM, dDW)            # DWc·∂_A(IKM)·∂_B(DW)
    T7h = Wc * dDW.T                                                         # ∂_B(DWc)·IKM·∂_A(DW)
    T8h = -jnp.einsum('i,ibc,cj,ja->ab', DWc, Gamma, IKM, dDW)            # DWc·∂_B(IKM)·∂_A(DW)
    T9h = jnp.einsum('i,ij,jab->ab', DWc, IKM, d2DW_hol)                   # DWc·IKM·∂_A∂_B(DW)

    d2S_AB = T1h + T2h + T3h + T4h + T5h + T6h + T7h + T8h + T9h

    # ══════════════════════════════════════════════════════════
    # G = W·W̄  derivatives (simple)
    # ══════════════════════════════════════════════════════════
    dW_A  = model.dW(z, tau, fluxes)         # ∂_I W, shape (n,)
    ddW   = model.ddW(z, tau, fluxes)        # ∂_I∂_J W, shape (n, n)

    dG_A  = dW_A * Wc                         # ∂_A G = (∂_A W) W̄
    dG_Bc = W * dWc_Bb                        # ∂_{B̄} G = W (∂_{B̄} W̄)
    d2G_ABc = jnp.outer(dW_A, dWc_Bb)        # ∂_A∂_{B̄} G = (∂_A W)(∂_{B̄} W̄)
    d2G_AB  = ddW * Wc                        # ∂_A∂_B G = (∂_A∂_B W) W̄

    # ══════════════════════════════════════════════════════════
    # Assemble F = S - λG  and its derivatives
    # ══════════════════════════════════════════════════════════
    S_val = DWc @ IKM @ DW
    F_val = S_val - lam * W * Wc

    dF_A    = dS_A    - lam * dG_A
    dF_Bc   = dS_Bc   - lam * dG_Bc
    d2F_ABc = d2S_ABc - lam * d2G_ABc
    d2F_AB  = d2S_AB  - lam * d2G_AB

    # ══════════════════════════════════════════════════════════
    # Product rule: V = e^K F
    # V_{AB̄} = e^K [K_A K_{B̄} F + K_{B̄} ∂_A F + K_{AB̄} F + K_A ∂_{B̄}F + ∂_A∂_{B̄}F]
    # V_{AB}  = e^K [K_A K_B  F + K_B  ∂_A F + K_{AB} F + K_A ∂_B F + ∂_A∂_B F]
    # ══════════════════════════════════════════════════════════
    V_mixed = eK * (jnp.outer(dK, dKc) * F_val + jnp.outer(dF_A, dKc)
                    + KM * F_val + jnp.outer(dK, dF_Bc) + d2F_ABc)

    V_holom = eK * (jnp.outer(dK, dK) * F_val + jnp.outer(dF_A, dK)
                    + KAB * F_val + jnp.outer(dK, dF_A) + d2F_AB)

    top = jnp.hstack([V_mixed, V_holom])
    bot = jnp.hstack([jnp.conj(V_holom), jnp.conj(V_mixed)])
    return jnp.vstack([top, bot])


# ── Verify ──
for ns, label in [(True, "noscale (λ=0)"), (False, "full (λ=3)")]:
    H_ref  = model._hessian_general(z, zc, tau, tauc, fl, noscale=ns)
    H_corr = hessian_SUGRA_corrected(model, z, zc, tau, tauc, fl, noscale=ns)
    err = float(jnp.max(jnp.abs(H_ref - H_corr)))
    scale = float(jnp.max(jnp.abs(H_ref)))
    print(f"  ✓ {label}: |err| = {err:.2e} (scale = {scale:.2e})")

z2_ = jnp.array([0.3 + 4.0j, -0.1 + 3.5j])
fl2_ = jnp.array([2., -1., 0., 1., -2., 3., 0., 1., -1., 2., 0., -1.])
tau2_ = 0.1 + 3.0j
for ns, label in [(True, "noscale"), (False, "full")]:
    Hr = model._hessian_general(z2_, jnp.conj(z2_), tau2_, jnp.conj(tau2_), fl2_, noscale=ns)
    Hc = hessian_SUGRA_corrected(model, z2_, jnp.conj(z2_), tau2_, jnp.conj(tau2_), fl2_, noscale=ns)
    print(f"  ✓ Point 2, {label}: |err| = {float(jnp.max(jnp.abs(Hr - Hc))):.2e}")

# Verify Christoffel and Riemann properties
Gamma = christoffel_symbols(model, z, zc, tau, tauc)
R = riemann_tensor(model, z, zc, tau, tauc)
print(f"\n  Γ torsion-free: |Γ^E_AC - Γ^E_CA| = {float(jnp.max(jnp.abs(Gamma - Gamma.transpose(0,2,1)))):.2e}")
print(f"  R symmetry R_ij̄kl̄ = R_kj̄il̄: {float(jnp.max(jnp.abs(R - R.transpose(2,1,0,3)))):.2e}")

## 9. Structure of the DW-dependent corrections

The corrections vanish at SUSY and are proportional to $D_A W$.
Let us verify this and show their magnitude relative to the full Hessian.

In [ ]:
# Show correction structure
H_ref = model._hessian_general(z, zc, tau, tauc, fl, noscale=True)

DDW_val = model.DDW(z, zc, tau, tauc, fl, mode=None)
W_val = model.superpotential(z, tau, fl)
KM_val = jnp.array(model.kahler_metric(z, zc, tau, tauc))
IKM_val = model.inverse_kahler_metric(z, zc, tau, tauc)
eK = jnp.exp(model.kahler_potential(z, zc, tau, tauc))

V_mixed_susy = eK * (DDW_val @ IKM_val.T @ jnp.conj(DDW_val) + KM_val * W_val * jnp.conj(W_val))
V_holom_susy = eK * 2. * DDW_val * jnp.conj(W_val)

corr_mixed = H_ref[:n, :n] - V_mixed_susy
corr_holom = H_ref[:n, n:] - V_holom_susy

print("DW-dependent corrections (noscale=True):")
print(f"  |correction to V_mixed| = {float(jnp.max(jnp.abs(corr_mixed))):.4e}")
print(f"  |correction to V_holom| = {float(jnp.max(jnp.abs(corr_holom))):.4e}")
print(f"  |V_mixed (SUSY part)|   = {float(jnp.max(jnp.abs(V_mixed_susy))):.4e}")
print(f"  |V_holom (SUSY part)|   = {float(jnp.max(jnp.abs(V_holom_susy))):.4e}")
print(f"  |DW|                    = {float(jnp.sum(jnp.abs(model.DW(z, zc, tau, tauc, fl)))):.4e}")
print()
print("The corrections are O(|DW|²) for V_mixed and O(|DW|) for V_holom.")
print("At a SUSY point (|DW| ~ 0), both corrections vanish.")

## 10. Summary of bugs and fixes

### Bug 1: `TypeError` — invalid keyword argument
**Lines 3039, 3052:** `self.DDW(..., check_SUSY=False)` — `DDW` does not have a `check_SUSY` parameter.

**Fix:** Remove `check_SUSY=False` from both calls.

### Bug 2: `print` inside JIT function

**Fix:** Remove the print statement.

### Bug 3: Missing DW-dependent terms in $V_{A\bar B}$
**Lines 3067–3069:** The mixed Hessian uses only the SUSY formula (terms (i) and (v) from Denef-Douglas).
The full formula at generic points includes three additional terms that vanish at $D_A W = 0$:

$$
\Delta V_{A\bar B} = e^K \bigl[
\underbrace{-R_{A\bar B C\bar D}\, K^{C\bar E}\, K^{F\bar D}\, F_F\, \bar F_{\bar E}}_{\text{Riemann curvature}}
+ \underbrace{K_{A\bar B}\, K^{C\bar D}\, F_C\, \bar F_{\bar D}}_{\text{metric} \times |F|^2}
+ \underbrace{F_A\, \bar F_{\bar B}}_{\text{F-term bilinear}}
\bigr]
$$

### Bug 4: Incomplete $V_{AB}$ (holomorphic Hessian)
**Lines 3071–3075:** The holomorphic Hessian has several DW-dependent terms partially implemented

The correct $V_{AB}$ at generic points involves derivatives of $M_{AB}$, $K^{-1}$,
and Christoffel connection terms. The full expression requires differentiating
the verified first-derivative formula

$$
\partial_A V = e^K \bigl[ K_A \cdot S + F_A \bar W + D_{\bar I}\bar W \cdot (\partial_A K^{I\bar J}) \cdot D_J W + D_{\bar I}\bar W \cdot K^{I\bar J} \cdot \partial_A(D_J W) \bigr]
$$

with respect to $z_B$ and $\tau$.

### Recommended fix strategy

The `_hessian_general` method (exact autodiff via `jacrev(dV)`) is **correct and reliable**.
For `hessian_SUGRA`, the cleanest fix is:

1. Fix bugs 1–2 (trivial)
2. For $V_{A\bar B}$: add the three missing DW-dependent terms (requires Riemann tensor computation)
3. For $V_{AB}$: complete the partial implementation or fall back to `jacrev(dV, argnums=(0,2))`

For practical use, `_hessian_general` should be preferred. The `hessian_SUGRA` method
is primarily useful for physical insight (decomposing the Hessian into SUGRA building blocks)
and for cross-validation.

## 11. Stress test: random moduli, dilaton, and flux choices

Verify `hessian_SUGRA_corrected` matches `_hessian_general` across many randomly chosen
field-space points and flux vectors. This rules out accidental agreement at specific test points.

In [ ]:
rng = np.random.default_rng(42)
n_tests = 50
max_rel_errors = []
max_abs_errors = []

print(f"Running {n_tests} random test points (h12={model.h12})...\n")
print(f"{'#':>3} {'Im(z1)':>7} {'Im(z2)':>7} {'Im(τ)':>7} {'|DW|':>10} {'noscale':>8} "
      f"{'|err|':>10} {'rel err':>10} {'ok?':>4}")
print("-" * 82)

for i in range(n_tests):
    # Random moduli: Re(z) ∈ [-0.5, 0.5], Im(z) ∈ [2, 5]
    re_z = rng.uniform(-0.5, 0.5, model.h12)
    im_z = rng.uniform(2.0, 5.0, model.h12)
    z_i  = jnp.array(re_z + 1j * im_z)
    zc_i = jnp.conj(z_i)
    
    # Random axio-dilaton: Re(τ) ∈ [-0.5, 0.5], Im(τ) ∈ [1, 8]
    tau_i  = complex(rng.uniform(-0.5, 0.5) + 1j * rng.uniform(1.0, 8.0))
    tauc_i = jnp.conj(tau_i)
    
    # Random integer fluxes ∈ [-5, 5]
    fl_i = jnp.array(rng.integers(-5, 6, size=2 * model.n_fluxes), dtype=float)
    
    # Skip if fluxes are all zero (trivial)
    if jnp.all(fl_i == 0):
        continue
    
    ns_i = bool(rng.choice([True, False]))
    
    H_ref_i  = model._hessian_general(z_i, zc_i, tau_i, tauc_i, fl_i, noscale=ns_i)
    H_corr_i = hessian_SUGRA_corrected(model, z_i, zc_i, tau_i, tauc_i, fl_i, noscale=ns_i)
    
    abs_err = float(jnp.max(jnp.abs(H_ref_i - H_corr_i)))
    scale_i = float(jnp.max(jnp.abs(H_ref_i)))
    rel_err = abs_err / scale_i if scale_i > 0 else 0.0
    
    max_rel_errors.append(rel_err)
    max_abs_errors.append(abs_err)
    
    DW_i = model.DW(z_i, zc_i, tau_i, tauc_i, fl_i)
    dw_norm = float(jnp.sum(jnp.abs(DW_i)))
    ok = "✓" if rel_err < 1e-10 else "✗"
    
    if i < 10 or rel_err > 1e-10:  # print first 10 + any failures
        print(f"{i:>3} {float(im_z[0]):>7.2f} {float(im_z[1]):>7.2f} "
              f"{float(jnp.imag(tau_i)):>7.2f} {dw_norm:>10.2e} "
              f"{'True' if ns_i else 'False':>8} "
              f"{abs_err:>10.2e} {rel_err:>10.2e} {ok:>4}")

max_rel = max(max_rel_errors)
max_abs = max(max_abs_errors)
n_pass = sum(1 for r in max_rel_errors if r < 1e-10)

print("-" * 82)
print(f"\nResults: {n_pass}/{len(max_rel_errors)} passed (rel err < 1e-10)")
print(f"  Max relative error: {max_rel:.2e}")
print(f"  Max absolute error: {max_abs:.2e}")
print(f"  Median rel error:   {np.median(max_rel_errors):.2e}")

if n_pass == len(max_rel_errors):
    print(f"\n✓  All {n_pass} random test points match to machine precision.")

## 12. Comparison of all Hessian implementations

The `FluxEFT` class provides four modes for computing the Hessian via `model.hessian(mode=...)`:

| Mode | Method | Valid at | Approach |
|---|---|---|---|
| `None` | `_hessian_general` | Generic points | `jacrev(dV)` — exact autodiff |
| `"SUSY"` | `_hessian_SUSY` | $D_AW = 0$ only | Analytical SUGRA formula (simplified at SUSY) |
| `"SUGRA"` | `_hessian_SUGRA` | Generic points | Explicit SUGRA building blocks ($\Gamma$, $R$, $D_IW$, $K$, $W$) |
| `"real"` | `ddV_x` | Generic points | Real-field Hessian via `jacrev` of real-field potential |

Below we compare all four at generic and SUSY points.

In [ ]:
import time

modes_all = [None, "SUSY", "SUGRA", "real"]

# ── Generic point (same as Section 2) ──
print("=" * 75)
print("A. GENERIC POINT (|DW| >> 0)")
print("=" * 75)

DW_gen = model.DW(z, zc, tau, tauc, fl)
print(f"|DW| = {float(jnp.sum(jnp.abs(DW_gen))):.2f}\n")

H_all = {}
timings = {}
for mode in modes_all:
    t0 = time.perf_counter()
    H_all[mode] = model.hessian(z, zc, tau, tauc, fl, noscale=True, mode=mode)
    # warm up, then time
    t0 = time.perf_counter()
    H_all[mode] = model.hessian(z, zc, tau, tauc, fl, noscale=True, mode=mode)
    jax.block_until_ready(H_all[mode])
    timings[mode] = time.perf_counter() - t0

# Reference: mode=None
H_ref = H_all[None]
scale = float(jnp.max(jnp.abs(H_ref)))

print(f"{'Mode':>8} {'Shape':>12} {'|err| vs None':>15} {'Rel err':>12} {'Time (ms)':>10}")
print("-" * 65)
for mode in modes_all:
    H = H_all[mode]
    
    if mode == "real":
        # Real Hessian has different layout — compare eigenvalues instead
        eig_ref = jnp.sort(jnp.linalg.eigvalsh(H_ref).real)
        eig_real = jnp.sort(jnp.linalg.eigvalsh(H).real)/2
        # The real Hessian is 2n×2n; the complex one is 2n×2n too but different basis
        err = float(jnp.max(jnp.abs(eig_ref - eig_real)))
        rel = err / scale if scale > 0 else 0.
        #sys.exit()
        #err_str = "(different basis)"
        #rel_str = "—"
    else:
        err = float(jnp.max(jnp.abs(H_ref - H)))
        rel = err / scale if scale > 0 else 0.
        
    err_str = f"{err:.2e}"
    rel_str = f"{rel:.2e}"
    
    ms = timings[mode] * 1000
    label = str(mode) if mode is not None else "None"
    print(f"{label:>8} {str(H.shape):>12} {err_str:>15} {rel_str:>12} {ms:>10.1f}")

# ── Also test noscale=False ──
print(f"\nnoscale=False (λ=3):")
for mode in modes_all:
    H_f = model.hessian(z, zc, tau, tauc, fl, noscale=False, mode=mode)
    H_ref_f = model.hessian(z, zc, tau, tauc, fl, noscale=False, mode=None)
    scale_f = float(jnp.max(jnp.abs(H_ref_f)))
    if mode=="real":
        eig_ref = jnp.sort(jnp.linalg.eigvalsh(H_ref_f).real)
        eig_real = jnp.sort(jnp.linalg.eigvalsh(H_f).real)/2
        # The real Hessian is 2n×2n; the complex one is 2n×2n too but different basis
        err = float(jnp.max(jnp.abs(eig_ref - eig_real)))
    else:
        err = float(jnp.max(jnp.abs(H_ref_f - H_f)))
        
    rel = err / scale_f if scale_f > 0 else 0.
    label = str(mode) if mode is not None else "None"
    print(f"  {label:>8}: rel err = {rel:.2e}")

# ── SUSY point ──
print("\n" + "=" * 75)
print("B. SUSY POINT (|DW| ≈ 0)")
print("=" * 75)

z_s = jnp.array([datsB[0,0]+1j*datsB[0,1], datsB[0,2]+1j*datsB[0,3]])
tau_s = datsB[0,4]+1j*datsB[0,5]
fl_s = jnp.array(datsB[0,6:18], dtype=float)
zc_s = jnp.conj(z_s); tauc_s = jnp.conj(tau_s)

DW_s = model.DW(z_s, zc_s, tau_s, tauc_s, fl_s)
print(f"|DW| = {float(jnp.sum(jnp.abs(DW_s))):.2e}\n")

H_ref_s = model.hessian(z_s, zc_s, tau_s, tauc_s, fl_s, noscale=True, mode=None)
scale_s = float(jnp.max(jnp.abs(H_ref_s)))

print(f"{'Mode':>8} {'Rel err vs None':>18} {'Status':>8}")
print("-" * 40)
for mode in modes_all:
    H_s = model.hessian(z_s, zc_s, tau_s, tauc_s, fl_s, noscale=True, mode=mode)
    if mode == "real":
        eig_ref = jnp.sort(jnp.linalg.eigvalsh(H_ref_s).real)
        eig_real = jnp.sort(jnp.linalg.eigvalsh(H_s).real)/2
        
        err = float(jnp.max(jnp.abs(eig_ref - eig_real)))
        rel = err / scale if scale > 0 else 0.
    else:
        err = float(jnp.max(jnp.abs(H_ref_s - H_s)))
        rel = err / scale_s if scale_s > 0 else 0.
    label = str(mode) if mode is not None else "None"
    status = "✓" if rel < 1e-10 else ("~" if rel < 1e-5 else "✗")
    print(f"{label:>8} {rel:>18.2e} {status:>8}")

# ── Stress test: all modes at 20 random points ──
print("\n" + "=" * 75)
print("C. STRESS TEST: 20 random points, all modes")
print("=" * 75)

rng2 = np.random.default_rng(123)
results = {m: [] for m in modes_all}

for i in range(20):
    z_ = jnp.array(rng2.uniform(-0.3, 0.3, 2) + 1j*rng2.uniform(2.5, 4.5, 2))
    tau_ = complex(rng2.uniform(-0.4, 0.4) + 1j*rng2.uniform(2, 7))
    fl_ = jnp.array(rng2.integers(-4, 5, 12), dtype=float)
    if jnp.all(fl_ == 0): fl_ = fl_.at[0].set(1.)
    zc_ = jnp.conj(z_); tauc_ = jnp.conj(tau_)
    ns_ = bool(rng2.choice([True, False]))
    
    H_ref_ = model.hessian(z_, zc_, tau_, tauc_, fl_, noscale=ns_, mode=None)
    scale_ = float(jnp.max(jnp.abs(H_ref_)))
    
    for mode in modes_all:
        H_ = model.hessian(z_, zc_, tau_, tauc_, fl_, noscale=ns_, mode=mode)
        if mode == "real":
            eig_ref = jnp.sort(jnp.linalg.eigvalsh(H_ref_).real)
            eig_real = jnp.sort(jnp.linalg.eigvalsh(H_).real)/2
            err = float(jnp.max(jnp.abs(eig_ref - eig_real)))
            rel_ = err / scale if scale > 0 else 0.
        else:
            rel_ = float(jnp.max(jnp.abs(H_ref_ - H_))) / scale_ if scale_ > 0 else 0.
        results[mode].append(rel_)

print(f"\n{'Mode':>8} {'Max rel err':>12} {'Median':>12} {'All < 1e-10':>12}")
print("-" * 50)
for mode in modes_all:
    errs = results[mode]
    label = str(mode) if mode is not None else "None"
    mx = max(errs)
    med = np.median(errs)
    ok = all(e < 1e-10 for e in errs)
    print(f"{label:>8} {mx:>12.2e} {med:>12.2e} {'✓' if ok else '✗':>12}")

print("\nNote: mode='SUSY' is only exact at DW=0; at generic points it has O(1) error.")